# MRI–JMA Williamson-5 T42/T63 reference preparation

This standalone notebook converts the published MRI–JMA/Yoshimura
`Williamson5/N959_1920x960/sh` grid fields into independently projected
T42 and T63 reference artifacts.

The only scientific path implemented here is:

\[
\text{published MRI grid fields}
\rightarrow \text{external spherical-harmonic analysis}
\rightarrow \text{triangular truncation}
\rightarrow \text{reference } h,u,v.
\]

It does not clone, install, import, inspect, or execute any comparison
model. It does not calculate comparison-model errors or declare a
benchmark result. All transform tolerances are fixed in the configuration
cell before source fields or projected results are inspected.

References:

- Yoshimura (2022), *Geoscientific Model Development*:
  https://doi.org/10.5194/gmd-15-2561-2022
- MRI–JMA public archive:
  https://climate.mri-jma.go.jp/pub/archives/Yoshimura_DFS_SW_Testcase/
- Williamson et al. (1992):
  https://doi.org/10.1016/S0021-9991(05)80016-6
- Skyborn spherical harmonics (NCAR SPHEREPACK backend):
  https://skyborn.readthedocs.io/en/latest/api/spharm.html


## 1. Mount Drive and configure the run

Edit this one cell if a different Drive location is desired. The exact
archive directory, source file sizes, required times, target grids,
library version, physical constants, and all validation tolerances are
fixed here.


In [ ]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DEFAULT_WORK_ROOT = Path("/content/drive/MyDrive/mri-w5-reference-preparation-v2")
else:
    # Local fallback is useful for an independent dry run of the notebook.
    DEFAULT_WORK_ROOT = Path(os.environ.get(
        "MRI_W5_WORK_ROOT",
        str(Path.cwd() / "mri-w5-reference-preparation-v2"),
    ))

WORK_ROOT = DEFAULT_WORK_ROOT
SOURCE_CACHE_DIR = WORK_ROOT / "source-cache" / "Williamson5_N959_1920x960_sh"
STAGE_DIR = WORK_ROOT / "stages"
OUTPUT_DIR = WORK_ROOT / "mri-w5-reference-v2"
SOURCE_INVENTORY_PATH = SOURCE_CACHE_DIR / "source_inventory.json"
VALIDATION_CHECKPOINT_PATH = STAGE_DIR / "validation_checkpoint.json"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

ARCHIVE_ROOT = "https://climate.mri-jma.go.jp/pub/archives/Yoshimura_DFS_SW_Testcase"
SOURCE_DIRECTORY_IDENTITY = "Williamson5/N959_1920x960/sh"
SOURCE_FILES = {
    "archive_README.txt": {
        "url": f"{ARCHIVE_ROOT}/README.txt",
        "expected_size": 5_197,
        "role": "archive README and directory/variable metadata",
    },
    "STDOUT": {
        "url": f"{ARCHIVE_ROOT}/{SOURCE_DIRECTORY_IDENTITY}/STDOUT",
        "expected_size": 6_007,
        "role": "model run metadata",
    },
    "data.nc": {
        "url": f"{ARCHIVE_ROOT}/{SOURCE_DIRECTORY_IDENTITY}/data.nc",
        "expected_size": 315_630_272,
        "role": "published MRI grid fields",
    },
    "data.ncctl": {
        "url": f"{ARCHIVE_ROOT}/{SOURCE_DIRECTORY_IDENTITY}/data.ncctl",
        "expected_size": 15_777,
        "role": "GrADS control metadata for data.nc",
    },
    "weight_lat.nc": {
        "url": f"{ARCHIVE_ROOT}/{SOURCE_DIRECTORY_IDENTITY}/weight_lat.nc",
        "expected_size": 30_469,
        "role": "published latitude weights",
    },
    "weight_lat.ncctl": {
        "url": f"{ARCHIVE_ROOT}/{SOURCE_DIRECTORY_IDENTITY}/weight_lat.ncctl",
        "expected_size": 15_600,
        "role": "GrADS control metadata for weight_lat.nc",
    },
}

REQUIRED_TIME_DAYS = (0.0, 5.0, 10.0, 15.0)
REQUIRED_TIME_INDICES = (0, 5, 10, 15)
SOURCE_GRID = {"nlat": 960, "nlon": 1920}
SOURCE_ANALYSIS_TRUNCATION = 958  # exact NMAX declared by the archived MRI STDOUT
TARGETS = {
    "t42_64x128.npz": {"truncation": 42, "nlat": 64, "nlon": 128},
    "t63_96x192.npz": {"truncation": 63, "nlat": 96, "nlon": 192},
}
EXPECTED_PROJECTED_ARRAY_HASHES = {
    "t42_64x128.npz": "fa320414507a7ed7f859c5061bd9782b07dda0b504ca5f68d57608360fb175d7",
    "t63_96x192.npz": "0072a566d0e8360277ef3da659dbffbc078d9fc3e6a7304c749d6d001f7a2dae",
}
V1_MANIFEST_SHA256 = "e294512a13ce89173eae2cda629bdb3f6d9c3322f0ecfa9293c9951589669c3d"

SKYBORN_VERSION = "0.4.2"
NETCDF4_VERSION = "1.7.4"
LEGENDRE_MODE = "computed"  # lower persistent memory at N959
PUBLIC_PRECISION = "double"
SOURCE_WEIGHT_SCALE = 2.0  # MRI wgt sums to 1; standard GL weights sum to 2

# Williamson et al. (1992), test case 5 constants; not taken from a comparison model.
WILLIAMSON = {
    "sphere_radius_m": 6.37122e6,
    "rotation_rate_s-1": 7.292e-5,
    "gravity_m_s-2": 9.80616,
    "u0_m_s-1": 20.0,
    "h0_m": 5960.0,
    "mountain_height_m": 2000.0,
    "mountain_radius_rad": 3.141592653589793 / 9.0,
    "mountain_lon_rad": 3.0 * 3.141592653589793 / 2.0,
    "mountain_lat_rad": 3.141592653589793 / 6.0,
}

# Fixed before source/projected-field inspection. Do not tune after seeing results.
TOLERANCES = {
    "source_latitude_max_abs_rad": 2.0e-10,
    "source_weight_raw_sum_abs": 5.0e-7,
    "source_weight_scaled_sum_abs": 5.0e-7,
    "source_weight_library_max_relative": 2.0e-6,
    "longitude_periodic_spacing_max_abs_rad": 5.0e-12,
    "analytic_day0_max_abs": {
        "layer_depth_m": 2.0e-3,
        "u_m_s-1": 5.0e-5,
        "v_m_s-1": 1.0e-10,
        "vor_s-1": 5.0e-11,
        "div_s-1": 5.0e-11,
    },
    "height_alternative_min_wrms_m": 10.0,
    "constant_scalar_max_abs": 1.0e-5,
    "synthetic_mode_coefficient_relative": 5.0e-6,
    "synthetic_mode_leakage_abs": 5.0e-6,
    "synthetic_phase_max_abs": 1.0e-5,
    "scalar_roundtrip_normalized_wrms": 5.0e-6,
    "scalar_roundtrip_max_abs": {
        "h": 5.0e-2,
        "vor": 2.0e-9,
        "div": 2.0e-10,
    },
    "velocity_reconstruction_max_abs_m_s-1": 2.0e-4,
    "velocity_reconstruction_normalized_wrms": 5.0e-6,
    "final_weight_sum_abs": 5.0e-13,
}

# Optional: set to a Drive path containing this .ipynb to include its byte hash.
NOTEBOOK_SOURCE_PATH = None

for directory in (SOURCE_CACHE_DIR, STAGE_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Work root: {WORK_ROOT}")
print(f"Exact source: {ARCHIVE_ROOT}/{SOURCE_DIRECTORY_IDENTITY}/")
print(f"Final package: {OUTPUT_DIR}")


## 2. Install the pinned external transform library

Skyborn provides a maintained Python interface to NCAR SPHEREPACK,
including scalar Gaussian-grid transforms and the library-owned
vorticity/divergence-to-wind reconstruction. Binary wheels are required;
this notebook will not replace those operations with local formulas.


In [ ]:
import importlib.metadata as metadata
import subprocess

required = {"skyborn": SKYBORN_VERSION, "netCDF4": NETCDF4_VERSION}
install_specs = []
for package, version in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install_specs.append(f"{package}=={version}")

if install_specs:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--only-binary=skyborn,netCDF4", *install_specs,
    ])

assert metadata.version("skyborn") == SKYBORN_VERSION
assert metadata.version("netCDF4") == NETCDF4_VERSION
print(f"External SH library ready: skyborn {SKYBORN_VERSION} (NCAR SPHEREPACK)")


## 3. Provenance, atomic I/O, and numerical helpers

Downloads use resumable `.part` files. A completed cache entry is reused
only after its recorded SHA-256 is recomputed and verified. JSON and NPZ
outputs are written to temporary files, flushed, and atomically renamed.


In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import re
import tempfile
import uuid

import numpy as np
import netCDF4
import requests
import skyborn
from skyborn.spharm import Spharmt, gaussian_lats_wts, getspecindx

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def jsonable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (datetime,)):
        return value.isoformat()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Not JSON serializable: {type(value)!r}")

def atomic_write_json(path, payload):
    path = Path(path)
    tmp = path.with_name(path.name + f".tmp-{uuid.uuid4().hex}")
    try:
        with tmp.open("w", encoding="utf-8", newline="\n") as stream:
            json.dump(
                payload, stream, indent=2, sort_keys=True,
                default=jsonable, allow_nan=False,
            )
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            tmp.unlink()

def atomic_save_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_name(path.name + f".tmp-{uuid.uuid4().hex}")
    try:
        with tmp.open("wb") as stream:
            np.savez_compressed(stream, **arrays)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            tmp.unlink()

def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while True:
            block = stream.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def reject_nonstandard_json_constant(token):
    raise ValueError(f"Non-standard JSON numeric constant: {token}")

def strict_json_load(stream):
    return json.load(stream, parse_constant=reject_nonstandard_json_constant)

def sha256_json(payload):
    raw = json.dumps(
        payload, sort_keys=True, separators=(",", ":"),
        default=jsonable, allow_nan=False,
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def require_finite_scalar(value, label):
    if value is None or not np.isfinite(value):
        raise RuntimeError(f"Non-finite validation value for {label}: {value!r}")
    return float(value)

def require_finite_tree(value, label="payload"):
    if isinstance(value, dict):
        for key, child in value.items():
            require_finite_tree(child, f"{label}.{key}")
    elif isinstance(value, (list, tuple)):
        for index, child in enumerate(value):
            require_finite_tree(child, f"{label}[{index}]")
    elif isinstance(value, (float, np.floating)) and not np.isfinite(value):
        raise RuntimeError(f"Non-finite numeric value at {label}: {value!r}")
    return value

def weighted_wrms(array, latitude_weights):
    values = np.asarray(array, dtype=np.float64)
    weights = np.asarray(latitude_weights, dtype=np.float64)
    if values.ndim != 2 or values.shape[0] != weights.size:
        raise ValueError("weighted_wrms requires [latitude, longitude]")
    if not np.all(np.isfinite(values)):
        raise RuntimeError("weighted_wrms received non-finite field values")
    if not np.all(np.isfinite(weights)) or np.any(weights <= 0.0):
        raise RuntimeError("weighted_wrms received invalid latitude weights")
    result = np.sqrt(
        np.sum(weights[:, None] * values * values)
        / (2.0 * values.shape[1])
    )
    return require_finite_scalar(result, "weighted WRMS")

def error_metrics(reconstructed, reference, latitude_weights):
    reconstructed = np.asarray(reconstructed, dtype=np.float64)
    reference = np.broadcast_to(
        np.asarray(reference, dtype=np.float64), reconstructed.shape
    )
    if not np.all(np.isfinite(reconstructed)):
        raise RuntimeError("Error metric reconstruction contains non-finite values")
    if not np.all(np.isfinite(reference)):
        raise RuntimeError("Error metric reference contains non-finite values")
    error = reconstructed - reference
    absolute_wrms = weighted_wrms(error, latitude_weights)
    reference_wrms = weighted_wrms(reference, latitude_weights)
    metrics = {
        "max_abs": require_finite_scalar(np.max(np.abs(error)), "maximum absolute error"),
        "wrms": absolute_wrms,
        "reference_wrms": reference_wrms,
        "normalized_wrms": (
            absolute_wrms / reference_wrms if reference_wrms > 0.0 else None
        ),
    }
    if metrics["normalized_wrms"] is not None:
        metrics["normalized_wrms"] = require_finite_scalar(
            metrics["normalized_wrms"], "normalized WRMS"
        )
    return metrics

def periodic_longitude_metrics(longitude):
    longitude = np.asarray(longitude, dtype=np.float64)
    wrapped = np.diff(np.concatenate([longitude, [longitude[0] + 2.0 * np.pi]]))
    nominal = 2.0 * np.pi / longitude.size
    return {
        "first_rad": float(longitude[0]),
        "last_rad": float(longitude[-1]),
        "nominal_spacing_rad": float(nominal),
        "max_spacing_error_rad": float(np.max(np.abs(wrapped - nominal))),
        "endpoint_excluded": bool(longitude[-1] < 2.0 * np.pi),
        "strictly_increasing": bool(np.all(np.diff(longitude) > 0.0)),
    }

def dependency_versions():
    names = ["skyborn", "numpy", "netCDF4", "requests", "cftime"]
    versions = {}
    for name in names:
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    return {
        "python": platform.python_version(),
        "implementation": platform.python_implementation(),
        "platform": platform.platform(),
        "packages": versions,
    }

print("Helpers loaded.")


## 4. Download and verify the exact MRI–JMA files

No alternative resolution, directory, or model variant is searched. A
size mismatch or URL failure is fatal. SHA-256 values established on the
first successful HTTPS download become the cache-verification record.


In [ ]:
def load_inventory():
    if SOURCE_INVENTORY_PATH.exists():
        with SOURCE_INVENTORY_PATH.open("r", encoding="utf-8") as stream:
            return strict_json_load(stream)
    return {
        "schema_version": 1,
        "dataset_identity": SOURCE_DIRECTORY_IDENTITY,
        "files": {},
    }

def download_atomic(session, local_name, spec, prior_entry):
    destination = SOURCE_CACHE_DIR / local_name
    expected_size = int(spec["expected_size"])

    if destination.exists() and prior_entry:
        if (
            prior_entry.get("url") == spec["url"]
            and prior_entry.get("size_bytes") == expected_size
            and destination.stat().st_size == expected_size
        ):
            actual_hash = sha256_file(destination)
            if actual_hash == prior_entry.get("sha256"):
                entry = dict(prior_entry)
                entry["last_cache_verification_utc"] = utc_now()
                return entry, True

    part = destination.with_name(destination.name + ".part")
    if part.exists() and part.stat().st_size > expected_size:
        part.unlink()

    resume_at = part.stat().st_size if part.exists() else 0
    headers = {"Range": f"bytes={resume_at}-"} if resume_at else {}
    with session.get(spec["url"], stream=True, timeout=(30, 180), headers=headers) as response:
        response.raise_for_status()
        if resume_at and response.status_code == 206:
            content_range = response.headers.get("Content-Range", "")
            if not content_range.startswith(f"bytes {resume_at}-"):
                raise RuntimeError(f"Unexpected Content-Range for {local_name}: {content_range}")
            mode = "ab"
        else:
            mode = "wb"
            resume_at = 0

        with part.open(mode) as stream:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    stream.write(chunk)
            stream.flush()
            os.fsync(stream.fileno())

    actual_size = part.stat().st_size
    if actual_size != expected_size:
        raise RuntimeError(
            f"{local_name}: expected {expected_size} bytes, downloaded {actual_size}"
        )
    digest = sha256_file(part)
    os.replace(part, destination)
    entry = {
        "local_name": local_name,
        "url": spec["url"],
        "role": spec["role"],
        "size_bytes": actual_size,
        "sha256": digest,
        "downloaded_at_utc": utc_now(),
        "last_cache_verification_utc": utc_now(),
    }
    return entry, False

source_inventory = load_inventory()
if source_inventory.get("dataset_identity") != SOURCE_DIRECTORY_IDENTITY:
    raise RuntimeError("Existing cache inventory identifies a different dataset")

with requests.Session() as session:
    session.headers.update({"User-Agent": "MRI-W5-reference-preparation/1.0"})
    for local_name, spec in SOURCE_FILES.items():
        entry, reused = download_atomic(
            session, local_name, spec, source_inventory["files"].get(local_name)
        )
        source_inventory["files"][local_name] = entry
        source_inventory["updated_at_utc"] = utc_now()
        atomic_write_json(SOURCE_INVENTORY_PATH, source_inventory)
        print(f"{'cache' if reused else 'download'}: {local_name} "
              f"({entry['size_bytes']} bytes, {entry['sha256'][:12]}...)")


## 5. Inspect the source schema and conventions

Variable semantics are established from the archive README, the control
files, and NetCDF metadata together—not from short variable names alone.
Missing values are scanned one time slice at a time to bound memory.


In [ ]:
def attr_value(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value

def variable_metadata(variable):
    return {
        "dimensions": list(variable.dimensions),
        "shape": list(variable.shape),
        "dtype": str(variable.dtype),
        "units": getattr(variable, "units", None),
        "long_name": getattr(variable, "long_name", None),
        "standard_name": getattr(variable, "standard_name", None),
        "fill_value": attr_value(getattr(variable, "_FillValue", None)),
        "missing_value": attr_value(getattr(variable, "missing_value", None)),
    }

def netcdf_metadata(path, scan_fields=()):
    with netCDF4.Dataset(path) as dataset:
        report = {
            "file_format": dataset.file_format,
            "dimensions": {name: len(dim) for name, dim in dataset.dimensions.items()},
            "global_attributes": {
                name: attr_value(getattr(dataset, name)) for name in dataset.ncattrs()
            },
            "variables": {
                name: variable_metadata(variable)
                for name, variable in dataset.variables.items()
            },
        }
        for name in scan_fields:
            variable = dataset.variables[name]
            masked_count = 0
            nonfinite_count = 0
            for time_index in range(variable.shape[0]):
                chunk = np.ma.asarray(variable[time_index])
                masked_count += int(np.ma.count_masked(chunk))
                filled = np.asarray(chunk.filled(np.nan), dtype=np.float64)
                nonfinite_count += int(np.count_nonzero(~np.isfinite(filled)))
            report["variables"][name]["masked_count_all_times"] = masked_count
            report["variables"][name]["nonfinite_count_all_times"] = nonfinite_count
    return report

readme_text = (SOURCE_CACHE_DIR / "archive_README.txt").read_text(encoding="utf-8")
stdout_text = (SOURCE_CACHE_DIR / "STDOUT").read_text(encoding="utf-8")
data_ctl_text = (SOURCE_CACHE_DIR / "data.ncctl").read_text(encoding="utf-8")
weight_ctl_text = (SOURCE_CACHE_DIR / "weight_lat.ncctl").read_text(encoding="utf-8")

required_text_evidence = [
    "Williamson5/        Williamson test case 5",
    "N959_1920x960/      N=959, IxJ=1920x960",
    "sh/                  SH model.",
    "Variables: h (height), u (zonal wind), v (meridional wind), vor (vorticity), div (divergence).",
]
for evidence in required_text_evidence:
    if evidence not in readme_text:
        raise RuntimeError(f"Required README evidence missing: {evidence!r}")

control_evidence = {
    "h": ("height", "(m)"),
    "u": ("zonal wind", "(m/s)"),
    "v": ("meridional wind", "(m/s)"),
    "vor": ("vorticity", "(1/s)"),
    "div": ("divergence", "(1/s)"),
}
for short_name, phrases in control_evidence.items():
    matching = [line for line in data_ctl_text.splitlines()
                if line.strip().startswith(short_name + " ")]
    if len(matching) != 1 or not all(phrase in matching[0] for phrase in phrases):
        raise RuntimeError(f"Ambiguous control-file metadata for {short_name!r}")

for token in ("JCN_INITIAL        =           5", "IMAX     =        1920",
              "JMAX     =         960", "NMAX     =         958"):
    if token not in stdout_text:
        raise RuntimeError(f"Required STDOUT metadata missing: {token!r}")

stdout_nmax_match = re.search(r"^\s*NMAX\s*=\s*(\d+)\s*$", stdout_text, re.MULTILINE)
stdout_truncation_match = re.search(
    r"NTRNC,MTRNC,NNUM,MNUM=\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
    stdout_text,
)
if stdout_nmax_match is None or stdout_truncation_match is None:
    raise RuntimeError("Could not parse NMAX or NTRNC/MTRNC/NNUM/MNUM from STDOUT")
stdout_nmax = int(stdout_nmax_match.group(1))
stdout_ntrnc, stdout_mtrnc, stdout_nnum, stdout_mnum = (
    int(value) for value in stdout_truncation_match.groups()
)
if stdout_ntrnc != stdout_nmax or stdout_mtrnc != stdout_nmax:
    raise RuntimeError(
        f"Unexpected MRI NMAX/NTRNC/MTRNC: {stdout_nmax}/"
        f"{stdout_ntrnc}/{stdout_mtrnc}"
    )
if stdout_nnum != stdout_nmax + 1:
    raise RuntimeError(f"Unexpected MRI NMAX/NNUM relationship: {stdout_nmax}/{stdout_nnum}")
if stdout_mnum != stdout_nnum:
    raise RuntimeError(f"Unexpected MRI NNUM/MNUM relationship: {stdout_nnum}/{stdout_mnum}")
if SOURCE_ANALYSIS_TRUNCATION != stdout_nmax:
    raise RuntimeError(
        "Source analysis truncation must equal the archived MRI NMAX: "
        f"configured={SOURCE_ANALYSIS_TRUNCATION}, STDOUT={stdout_nmax}"
    )

data_metadata = netcdf_metadata(
    SOURCE_CACHE_DIR / "data.nc", scan_fields=("h", "u", "v", "vor", "div")
)
weight_metadata = netcdf_metadata(
    SOURCE_CACHE_DIR / "weight_lat.nc", scan_fields=("wgt",)
)

if data_metadata["dimensions"] != {"time": 16, "lon": 1920, "lat": 960}:
    raise RuntimeError(f"Unexpected data.nc dimensions: {data_metadata['dimensions']}")
if weight_metadata["dimensions"] != {"time": 1, "lon": 1, "lat": 960}:
    raise RuntimeError(f"Unexpected weight_lat.nc dimensions: {weight_metadata['dimensions']}")

with netCDF4.Dataset(SOURCE_CACHE_DIR / "data.nc") as dataset:
    source_time_raw = np.asarray(dataset.variables["time"][:], dtype=np.float64)
    source_time_units = dataset.variables["time"].units
    source_time_calendar = dataset.variables["time"].calendar
    decoded_time = netCDF4.num2date(
        source_time_raw, source_time_units, calendar=source_time_calendar
    )
    source_time_days = (source_time_raw - source_time_raw[0]) / 24.0
    source_latitude_deg_south_to_north = np.asarray(
        dataset.variables["lat"][:], dtype=np.float64
    )
    source_longitude_deg = np.asarray(dataset.variables["lon"][:], dtype=np.float64)

expected_days = np.arange(16, dtype=np.float64)
if not np.array_equal(source_time_days, expected_days):
    raise RuntimeError(f"Unexpected daily time coordinate: {source_time_days}")
if tuple(int(np.flatnonzero(source_time_days == day)[0])
         for day in REQUIRED_TIME_DAYS) != REQUIRED_TIME_INDICES:
    raise RuntimeError("Required day-to-index mapping is not exactly (0, 5, 10, 15)")

if not np.all(np.diff(source_latitude_deg_south_to_north) > 0.0):
    raise RuntimeError("Published latitude is not strictly south-to-north")
if not np.all(np.diff(source_longitude_deg) > 0.0):
    raise RuntimeError("Published longitude is not strictly eastward")

longitude_rad = np.deg2rad(source_longitude_deg)
source_periodicity = periodic_longitude_metrics(longitude_rad)
if (
    source_periodicity["max_spacing_error_rad"]
    > TOLERANCES["longitude_periodic_spacing_max_abs_rad"]
    or not source_periodicity["endpoint_excluded"]
):
    raise RuntimeError(f"Source longitude periodicity failure: {source_periodicity}")

with netCDF4.Dataset(SOURCE_CACHE_DIR / "weight_lat.nc") as weights_dataset:
    weight_latitudes = np.asarray(weights_dataset.variables["lat"][:], dtype=np.float64)
    source_weights_raw_south_to_north = np.asarray(
        weights_dataset.variables["wgt"][0, :, 0], dtype=np.float64
    )
if not np.array_equal(weight_latitudes, source_latitude_deg_south_to_north):
    raise RuntimeError("weight_lat.nc latitude does not exactly equal data.nc latitude")

for report in (data_metadata, weight_metadata):
    for name, variable in report["variables"].items():
        if variable.get("masked_count_all_times", 0) != 0:
            raise RuntimeError(f"Masked values found in {name}")
        if variable.get("nonfinite_count_all_times", 0) != 0:
            raise RuntimeError(f"Non-finite values found in {name}")

source_variable_mapping = {
    "height": {
        "netcdf_variable": "h",
        "units": "m",
        "evidence": ["archive README: h (height)", "data.ncctl: height (m)",
                     f"NetCDF long_name: {data_metadata['variables']['h']['long_name']}"],
    },
    "zonal_velocity": {
        "netcdf_variable": "u",
        "units": "m s-1",
        "positive_direction": "eastward",
        "evidence": ["archive README: u (zonal wind)", "data.ncctl: zonal wind (m/s)",
                     f"NetCDF long_name: {data_metadata['variables']['u']['long_name']}"],
    },
    "meridional_velocity": {
        "netcdf_variable": "v",
        "units": "m s-1",
        "positive_direction": "northward",
        "evidence": ["archive README: v (meridional wind)",
                     "data.ncctl: meridional wind (m/s)",
                     f"NetCDF long_name: {data_metadata['variables']['v']['long_name']}"],
    },
    "relative_vorticity": {
        "netcdf_variable": "vor",
        "units": "s-1",
        "evidence": ["archive README: vor (vorticity)",
                     "data.ncctl: vorticity (1/s)",
                     f"NetCDF long_name: {data_metadata['variables']['vor']['long_name']}"],
    },
    "horizontal_divergence": {
        "netcdf_variable": "div",
        "units": "s-1",
        "evidence": ["archive README: div (divergence)",
                     "data.ncctl: divergence (1/s)",
                     f"NetCDF long_name: {data_metadata['variables']['div']['long_name']}"],
    },
}

source_inspection = {
    "variables": data_metadata["variables"],
    "time": {
        "units": source_time_units,
        "calendar": source_time_calendar,
        "raw_values": source_time_raw.tolist(),
        "decoded_values": [str(value) for value in decoded_time],
        "days_since_first_output": source_time_days.tolist(),
        "required_indices": list(REQUIRED_TIME_INDICES),
    },
    "latitude": {
        "orientation_in_file": "south_to_north",
        "minimum_degrees_north": float(source_latitude_deg_south_to_north[0]),
        "maximum_degrees_north": float(source_latitude_deg_south_to_north[-1]),
    },
    "longitude": {
        "range": "[0, 360) degrees_east",
        "ordering": "strictly increasing eastward",
        "duplicate_periodic_endpoint": False,
        **source_periodicity,
    },
    "missing_values": {
        name: {
            "fill_value": data_metadata["variables"][name]["fill_value"],
            "missing_value": data_metadata["variables"][name]["missing_value"],
            "masked_count_all_times": data_metadata["variables"][name]["masked_count_all_times"],
            "nonfinite_count_all_times": data_metadata["variables"][name]["nonfinite_count_all_times"],
        }
        for name in ("h", "u", "v", "vor", "div")
    },
    "data_weight_relationship": {
        "same_latitude_coordinate_exactly": True,
        "wgt_dimensions": weight_metadata["variables"]["wgt"]["dimensions"],
        "raw_weight_sum": float(source_weights_raw_south_to_north.sum()),
        "raw_weight_normalization": "hemisphere-normalized; multiply by 2 for sum=2",
    },
    "spectral_configuration": {
        "archive_label": "N959",
        "stdout_nmax": stdout_nmax,
        "stdout_ntrnc": stdout_ntrnc,
        "stdout_mtrnc": stdout_mtrnc,
        "stdout_nnum": stdout_nnum,
        "stdout_mnum": stdout_mnum,
        "analysis_truncation": SOURCE_ANALYSIS_TRUNCATION,
    },
}

for local_name, metadata_report in (
    ("data.nc", data_metadata), ("weight_lat.nc", weight_metadata)
):
    source_inventory["files"][local_name]["netcdf"] = metadata_report
source_inventory["updated_at_utc"] = utc_now()
atomic_write_json(SOURCE_INVENTORY_PATH, source_inventory)

print("Source variables:")
for name, info in data_metadata["variables"].items():
    print(f"  {name}: dims={tuple(info['dimensions'])}, dtype={info['dtype']}, "
          f"units={info['units']!r}, long_name={info['long_name']!r}, "
          f"standard_name={info['standard_name']!r}")
print(f"Time: {source_time_units}; days={source_time_days.tolist()}")
print("Latitude: south-to-north in source; will be reversed for transforms/output")
print(f"Longitude: [0, 360), endpoint excluded; periodic spacing error "
      f"{source_periodicity['max_spacing_error_rad']:.3e} rad")
print(f"weight_lat relationship: identical latitude; raw sum "
      f"{source_weights_raw_south_to_north.sum():.12f}")


## 6. Resolve `h` and verify the analytic day-zero state

Williamson-5 uses the balanced zonal-flow layer depth from test case 2
over a separate conical bottom topography. Two physical fields are checked
explicitly:

1. `layer_depth`: the canonical smooth balanced W5 fluid-layer thickness;
2. `free_surface_elevation`: `layer_depth + bottom_topography`.

MRI `h` must match `layer_depth`; the free-surface candidate must differ
by the isolated mountain. Day-zero `u`,
`v`, relative vorticity, and divergence are checked at the same time.


In [ ]:
source_latitude_rad = np.deg2rad(source_latitude_deg_south_to_north[::-1])
source_longitude_rad = np.deg2rad(source_longitude_deg)
source_weights = SOURCE_WEIGHT_SCALE * source_weights_raw_south_to_north[::-1]

phi = source_latitude_rad[:, None]
lam = source_longitude_rad[None, :]
a = WILLIAMSON["sphere_radius_m"]
omega = WILLIAMSON["rotation_rate_s-1"]
gravity = WILLIAMSON["gravity_m_s-2"]
u0 = WILLIAMSON["u0_m_s-1"]
h0 = WILLIAMSON["h0_m"]

balanced_drop_m = (a * omega * u0 + 0.5 * u0 * u0) / gravity
analytic_layer_depth = h0 - balanced_drop_m * np.sin(phi) ** 2

delta_lon = (
    lam - WILLIAMSON["mountain_lon_rad"] + np.pi
) % (2.0 * np.pi) - np.pi
mountain_distance = np.minimum(
    WILLIAMSON["mountain_radius_rad"],
    np.sqrt(delta_lon ** 2 + (phi - WILLIAMSON["mountain_lat_rad"]) ** 2),
)
bottom_topography = WILLIAMSON["mountain_height_m"] * (
    1.0 - mountain_distance / WILLIAMSON["mountain_radius_rad"]
)
analytic_free_surface_elevation = analytic_layer_depth + bottom_topography
analytic_u = u0 * np.cos(phi)
analytic_v = np.zeros_like(phi)
analytic_vorticity = 2.0 * u0 * np.sin(phi) / a
analytic_divergence = np.zeros_like(phi)

with netCDF4.Dataset(SOURCE_CACHE_DIR / "data.nc") as dataset:
    day0 = {
        name: np.asarray(dataset.variables[name][0, ::-1, :], dtype=np.float64)
        for name in ("h", "u", "v", "vor", "div")
    }

height_candidates = {
    "layer_depth": error_metrics(day0["h"], analytic_layer_depth, source_weights),
    "free_surface_elevation": error_metrics(
        day0["h"], analytic_free_surface_elevation, source_weights
    ),
}
layer_depth_limit = TOLERANCES["analytic_day0_max_abs"]["layer_depth_m"]
passing_height_candidates = [
    name for name, metrics in height_candidates.items()
    if metrics["max_abs"] <= layer_depth_limit
]
if passing_height_candidates != ["layer_depth"]:
    raise RuntimeError(
        "Height semantics are not uniquely resolved: "
        + json.dumps(height_candidates, indent=2)
    )
resolved_height_interpretation = passing_height_candidates[0]
rejected_height = "free_surface_elevation"
if (
    height_candidates[rejected_height]["wrms"]
    < TOLERANCES["height_alternative_min_wrms_m"]
):
    raise RuntimeError("Rejected height interpretation is not sufficiently separated")

analytic_day0_metrics = {
    "h": height_candidates[resolved_height_interpretation],
    "u": error_metrics(day0["u"], analytic_u, source_weights),
    "v": error_metrics(day0["v"], analytic_v, source_weights),
    "vor": error_metrics(day0["vor"], analytic_vorticity, source_weights),
    "div": error_metrics(day0["div"], analytic_divergence, source_weights),
    "height_candidates": height_candidates,
}
analytic_limits = TOLERANCES["analytic_day0_max_abs"]
analytic_checks = {
    "h": analytic_day0_metrics["h"]["max_abs"] <= analytic_limits["layer_depth_m"],
    "u": analytic_day0_metrics["u"]["max_abs"] <= analytic_limits["u_m_s-1"],
    "v": analytic_day0_metrics["v"]["max_abs"] <= analytic_limits["v_m_s-1"],
    "vor": analytic_day0_metrics["vor"]["max_abs"] <= analytic_limits["vor_s-1"],
    "div": analytic_day0_metrics["div"]["max_abs"] <= analytic_limits["div_s-1"],
}
if not all(analytic_checks.values()):
    raise RuntimeError(
        "Published day-zero fields do not match the Williamson-5 analytic state: "
        + json.dumps(analytic_day0_metrics, indent=2)
    )

run_signature_payload = {
    "schema": "mri-w5-reference-preparation-v2",
    "source_directory": SOURCE_DIRECTORY_IDENTITY,
    "source_hashes": {
        name: source_inventory["files"][name]["sha256"] for name in SOURCE_FILES
    },
    "source_sizes": {
        name: source_inventory["files"][name]["size_bytes"] for name in SOURCE_FILES
    },
    "required_time_days": REQUIRED_TIME_DAYS,
    "required_time_indices": REQUIRED_TIME_INDICES,
    "source_analysis_truncation": SOURCE_ANALYSIS_TRUNCATION,
    "targets": TARGETS,
    "skyborn_version": SKYBORN_VERSION,
    "netcdf4_version": NETCDF4_VERSION,
    "legendre_mode": LEGENDRE_MODE,
    "public_precision": PUBLIC_PRECISION,
    "source_weight_scale": SOURCE_WEIGHT_SCALE,
    "williamson_constants": WILLIAMSON,
    "tolerances": TOLERANCES,
    "height_interpretation": resolved_height_interpretation,
}
run_signature = sha256_json(run_signature_payload)

print(f"Resolved h: {resolved_height_interpretation}")
print("Day-zero maximum absolute errors: "
      + ", ".join(
          f"{name}={analytic_day0_metrics[name]['max_abs']:.3e}"
          for name in ("h", "u", "v", "vor", "div")
      ))


## 7. Transform and convention validation

The published grid is reversed to north-to-south, as required by the
external library. MRI's half-normalized weights are scaled by the fixed
factor 2 and must match Skyborn/SPHEREPACK's Gaussian quadrature weights.

Tests below cover constant preservation, zonal and non-zonal modes, odd
order, complex phase, modes at the T42/T63 limits, longitude periodicity,
latitude orientation, full-resolution scalar round trips, and full
vorticity/divergence-to-wind reconstruction against the published winds.
No local inverse Laplacian, derivative recurrence, vector basis
conversion, or pole treatment is implemented.


In [ ]:
source_sht = Spharmt(
    SOURCE_GRID["nlon"], SOURCE_GRID["nlat"],
    rsphere=WILLIAMSON["sphere_radius_m"],
    gridtype="gaussian", legfunc=LEGENDRE_MODE,
    precision=PUBLIC_PRECISION,
)
target_sht = {
    filename: Spharmt(
        spec["nlon"], spec["nlat"],
        rsphere=WILLIAMSON["sphere_radius_m"],
        gridtype="gaussian", legfunc=LEGENDRE_MODE,
        precision=PUBLIC_PRECISION,
    )
    for filename, spec in TARGETS.items()
}

library_source_latitude_deg, library_source_weights = gaussian_lats_wts(
    SOURCE_GRID["nlat"]
)
library_source_latitude_rad = np.deg2rad(library_source_latitude_deg)
latitude_max_abs = float(np.max(
    np.abs(source_latitude_rad - library_source_latitude_rad)
))
raw_weight_sum_error = abs(float(source_weights_raw_south_to_north.sum()) - 1.0)
scaled_weight_sum_error = abs(float(source_weights.sum()) - 2.0)
weight_max_relative = float(np.max(
    np.abs(source_weights - library_source_weights) / library_source_weights
))

if latitude_max_abs > TOLERANCES["source_latitude_max_abs_rad"]:
    raise RuntimeError("Source latitude does not match the library Gaussian grid")
if raw_weight_sum_error > TOLERANCES["source_weight_raw_sum_abs"]:
    raise RuntimeError("MRI source weights do not have the documented observed sum=1")
if scaled_weight_sum_error > TOLERANCES["source_weight_scaled_sum_abs"]:
    raise RuntimeError("Scaled MRI source weights do not sum to 2")
if weight_max_relative > TOLERANCES["source_weight_library_max_relative"]:
    raise RuntimeError("Scaled MRI weights do not match library Gaussian weights")
if not np.all(np.diff(library_source_latitude_rad) < 0.0):
    raise RuntimeError("Library/source working latitude is not north-to-south")

target_coordinates = {}
for filename, spec in TARGETS.items():
    latitude_deg, weights = gaussian_lats_wts(spec["nlat"])
    latitude = np.deg2rad(latitude_deg)
    longitude = 2.0 * np.pi * np.arange(spec["nlon"]) / spec["nlon"]
    periodicity = periodic_longitude_metrics(longitude)
    if not np.all(np.diff(latitude) < 0.0):
        raise RuntimeError(f"{filename}: target latitude is not north-to-south")
    if abs(float(weights.sum()) - 2.0) > TOLERANCES["final_weight_sum_abs"]:
        raise RuntimeError(f"{filename}: target Gaussian weights do not sum to 2")
    if (
        periodicity["max_spacing_error_rad"]
        > TOLERANCES["longitude_periodic_spacing_max_abs_rad"]
        or not periodicity["endpoint_excluded"]
    ):
        raise RuntimeError(f"{filename}: target longitude periodicity failure")
    target_coordinates[filename] = {
        "latitude": latitude.astype(np.float64),
        "longitude": longitude.astype(np.float64),
        "latitude_weights": np.asarray(weights, dtype=np.float64),
        "periodicity": periodicity,
    }

source_m, source_n = getspecindx(SOURCE_ANALYSIS_TRUNCATION)

def truncate_spectrum(full_spectrum, truncation):
    full_spectrum = np.asarray(full_spectrum)
    if full_spectrum.shape[0] != source_n.size:
        raise ValueError("Unexpected full-resolution spectral shape")
    if not np.all(np.isfinite(full_spectrum)):
        raise RuntimeError("Full-resolution spectrum contains non-finite values")
    mask = source_n <= truncation
    target_m, target_n = getspecindx(truncation)
    if not (
        np.array_equal(source_m[mask], target_m)
        and np.array_equal(source_n[mask], target_n)
    ):
        raise RuntimeError("Packed triangular ordering changed during truncation")
    # Pure restriction: select n <= truncation; no coefficient modification.
    truncated = full_spectrum[mask].copy()
    if not np.all(np.isfinite(truncated)):
        raise RuntimeError("Truncated spectrum contains non-finite values")
    return truncated

def constant_test(sht, nlat, nlon, truncation):
    field = np.full((nlat, nlon), 3.25, dtype=np.float64)
    spectrum = sht.grdtospec(field, ntrunc=truncation)
    if not np.all(np.isfinite(spectrum)):
        raise RuntimeError(
            f"Constant-field analysis returned non-finite coefficients at T{truncation}"
        )
    reconstructed = sht.spectogrd(spectrum)
    if not np.all(np.isfinite(reconstructed)):
        raise RuntimeError(
            f"Constant-field synthesis returned non-finite values at T{truncation}"
        )
    return {
        "max_abs": require_finite_scalar(
            np.max(np.abs(reconstructed - field)),
            f"constant-field T{truncation} maximum absolute error",
        )
    }

def synthetic_mode_tests(sht, nlat, nlon, truncation):
    modes_m, modes_n = getspecindx(truncation)
    requested = [
        (8, 0, complex(1.0, 0.0), "zonal_m0"),
        (17, 5, complex(1.0, 0.37), "odd_m_complex_phase"),
        (truncation, truncation - 1, complex(-0.6, 0.8), "near_limit"),
        (truncation, truncation, complex(0.35, -0.91), "at_limit"),
    ]
    results = []
    for degree, order, coefficient, label in requested:
        location = np.flatnonzero((modes_m == order) & (modes_n == degree))
        if location.size != 1:
            raise RuntimeError(f"Cannot locate synthetic mode {(degree, order)}")
        spectrum = np.zeros(modes_m.size, dtype=np.complex128)
        spectrum[location[0]] = coefficient
        field = sht.spectogrd(spectrum)
        if not np.all(np.isfinite(field)):
            raise RuntimeError(f"Synthetic mode synthesis is non-finite: {label}")
        recovered = sht.grdtospec(field, ntrunc=truncation)
        if not np.all(np.isfinite(recovered)):
            raise RuntimeError(f"Synthetic mode analysis is non-finite: {label}")
        coefficient_error = np.abs(recovered - spectrum)
        off_mode = np.delete(np.abs(recovered), location[0])
        metrics = {
            "label": label,
            "degree_n": degree,
            "order_m": order,
            "coefficient_real": coefficient.real,
            "coefficient_imag": coefficient.imag,
            "coefficient_relative_error": require_finite_scalar(
                coefficient_error.max() / abs(coefficient),
                f"{label} coefficient relative error",
            ),
            "off_mode_leakage_abs": require_finite_scalar(
                off_mode.max(initial=0.0), f"{label} off-mode leakage"
            ),
        }
        if (
            metrics["coefficient_relative_error"]
            > TOLERANCES["synthetic_mode_coefficient_relative"]
            or metrics["off_mode_leakage_abs"]
            > TOLERANCES["synthetic_mode_leakage_abs"]
        ):
            raise RuntimeError(f"Synthetic mode failure: {metrics}")
        results.append(metrics)

    # Explicit SPHEREPACK phase check. Skyborn stores c=a+i*b and
    # synthesizes a*cos(m*lambda)-b*sin(m*lambda).
    degree, order = 17, 5
    location = np.flatnonzero((modes_m == order) & (modes_n == degree))
    real_spec = np.zeros(modes_m.size, dtype=np.complex128)
    imag_spec = np.zeros(modes_m.size, dtype=np.complex128)
    real_spec[location[0]] = 1.0
    imag_spec[location[0]] = 1.0j
    real_field = sht.spectogrd(real_spec)
    imag_field = sht.spectogrd(imag_spec)
    longitude = 2.0 * np.pi * np.arange(nlon) / nlon
    amplitude = real_field[:, :1]
    real_expected = amplitude * np.cos(order * longitude)[None, :]
    imag_expected = -amplitude * np.sin(order * longitude)[None, :]
    phase_max_abs = require_finite_scalar(max(
        np.max(np.abs(real_field - real_expected)),
        np.max(np.abs(imag_field - imag_expected)),
    ), "synthetic longitude-phase maximum absolute error")
    if phase_max_abs > TOLERANCES["synthetic_phase_max_abs"]:
        raise RuntimeError(f"SPHEREPACK longitude phase failure: {phase_max_abs}")
    return results, {
        "storage": "complex coefficient c = a + i*b",
        "synthesis_phase": "a*cos(m*lambda) - b*sin(m*lambda)",
        "max_abs": phase_max_abs,
    }

quick_validation = {
    "source_grid": {
        "latitude_max_abs_rad": latitude_max_abs,
        "raw_weight_sum": float(source_weights_raw_south_to_north.sum()),
        "scaled_weight_sum": float(source_weights.sum()),
        "scaled_weight_library_max_relative": weight_max_relative,
        "latitude_orientation_working": "north_to_south",
        "longitude_periodicity": source_periodicity,
    },
    "constant_scalar": {},
    "synthetic_modes": {},
    "phase": {},
    "target_grids": {},
}

quick_validation["constant_scalar"]["source_N959_grid"] = constant_test(
    source_sht, 960, 1920, SOURCE_ANALYSIS_TRUNCATION
)
if (
    quick_validation["constant_scalar"]["source_N959_grid"]["max_abs"]
    > TOLERANCES["constant_scalar_max_abs"]
):
    raise RuntimeError("Full-grid constant scalar preservation failed")

for filename, spec in TARGETS.items():
    constant_metrics = constant_test(
        target_sht[filename], spec["nlat"], spec["nlon"], spec["truncation"]
    )
    mode_metrics, phase_metrics = synthetic_mode_tests(
        target_sht[filename], spec["nlat"], spec["nlon"], spec["truncation"]
    )
    if constant_metrics["max_abs"] > TOLERANCES["constant_scalar_max_abs"]:
        raise RuntimeError(f"{filename}: constant scalar preservation failed")
    quick_validation["constant_scalar"][filename] = constant_metrics
    quick_validation["synthetic_modes"][filename] = mode_metrics
    quick_validation["phase"][filename] = phase_metrics
    quick_validation["target_grids"][filename] = {
        "truncation": spec["truncation"],
        "nlat": spec["nlat"],
        "nlon": spec["nlon"],
        "latitude_orientation": "north_to_south",
        "longitude": target_coordinates[filename]["periodicity"],
        "latitude_weight_sum": float(
            target_coordinates[filename]["latitude_weights"].sum()
        ),
    }

print(f"Grid/weight equivalence: dlat={latitude_max_abs:.3e} rad, "
      f"max relative weight error={weight_max_relative:.3e}")
print("Constant, single-mode, complex-phase, orientation, and periodicity checks passed.")


In [ ]:
checkpoint = None
if VALIDATION_CHECKPOINT_PATH.exists():
    try:
        with VALIDATION_CHECKPOINT_PATH.open("r", encoding="utf-8") as stream:
            candidate = strict_json_load(stream)
        require_finite_tree(candidate, "cached_validation_checkpoint")
        if (
            candidate.get("run_signature") == run_signature
            and candidate.get("all_reference_preparation_checks_passed") is True
        ):
            checkpoint = candidate
    except (ValueError, RuntimeError, json.JSONDecodeError) as exc:
        print(f"Ignoring invalid validation checkpoint; it will be recomputed: {exc}")

if checkpoint is None:
    scalar_roundtrip_metrics = []
    velocity_reconstruction_metrics = []

    with netCDF4.Dataset(SOURCE_CACHE_DIR / "data.nc") as dataset:
        for time_index, time_day in zip(
            REQUIRED_TIME_INDICES, REQUIRED_TIME_DAYS
        ):
            print(f"Validating full-resolution day {time_day:g}...")
            for variable_name in ("h", "vor", "div"):
                field = np.asarray(
                    dataset.variables[variable_name][time_index, ::-1, :],
                    dtype=np.float64,
                )
                if not np.all(np.isfinite(field)):
                    raise RuntimeError(
                        f"Non-finite source values in {variable_name}, day {time_day}"
                    )
                spectrum = source_sht.grdtospec(
                    field, ntrunc=SOURCE_ANALYSIS_TRUNCATION
                )
                reconstructed = source_sht.spectogrd(spectrum)
                metrics = error_metrics(reconstructed, field, source_weights)
                metrics.update({
                    "time_index": time_index,
                    "time_day": time_day,
                    "variable": variable_name,
                })
                normalized = metrics["normalized_wrms"]
                if (
                    metrics["max_abs"]
                    > TOLERANCES["scalar_roundtrip_max_abs"][variable_name]
                    or normalized is None
                    or normalized
                    > TOLERANCES["scalar_roundtrip_normalized_wrms"]
                ):
                    raise RuntimeError(
                        "Full-resolution scalar round trip failed: "
                        + json.dumps(metrics, indent=2)
                    )
                scalar_roundtrip_metrics.append(metrics)
                del field, spectrum, reconstructed

            vorticity = np.asarray(
                dataset.variables["vor"][time_index, ::-1, :], dtype=np.float64
            )
            divergence = np.asarray(
                dataset.variables["div"][time_index, ::-1, :], dtype=np.float64
            )
            vorticity_spectrum = source_sht.grdtospec(
                vorticity, ntrunc=SOURCE_ANALYSIS_TRUNCATION
            )
            divergence_spectrum = source_sht.grdtospec(
                divergence, ntrunc=SOURCE_ANALYSIS_TRUNCATION
            )
            reconstructed_u, reconstructed_v = source_sht.getuv(
                vorticity_spectrum, divergence_spectrum
            )
            published_u = np.asarray(
                dataset.variables["u"][time_index, ::-1, :], dtype=np.float64
            )
            published_v = np.asarray(
                dataset.variables["v"][time_index, ::-1, :], dtype=np.float64
            )
            published_speed = np.hypot(published_u, published_v)
            reconstructed_speed = np.hypot(reconstructed_u, reconstructed_v)
            speed_scale = weighted_wrms(published_speed, source_weights)

            component_metrics = {}
            for name, reconstructed, published in (
                ("u", reconstructed_u, published_u),
                ("v", reconstructed_v, published_v),
                ("speed", reconstructed_speed, published_speed),
            ):
                metrics = error_metrics(reconstructed, published, source_weights)
                metrics["normalized_by_published_speed_wrms"] = (
                    metrics["wrms"] / speed_scale
                )
                if (
                    metrics["max_abs"]
                    > TOLERANCES["velocity_reconstruction_max_abs_m_s-1"]
                    or metrics["normalized_by_published_speed_wrms"]
                    > TOLERANCES["velocity_reconstruction_normalized_wrms"]
                ):
                    raise RuntimeError(
                        f"Full-resolution velocity reconstruction failed "
                        f"for {name}, day {time_day}: {metrics}"
                    )
                component_metrics[name] = metrics
            velocity_reconstruction_metrics.append({
                "time_index": time_index,
                "time_day": time_day,
                "published_speed_wrms_m_s-1": speed_scale,
                "components": component_metrics,
            })
            del (
                vorticity, divergence, vorticity_spectrum, divergence_spectrum,
                reconstructed_u, reconstructed_v, published_u, published_v,
                published_speed, reconstructed_speed,
            )

    validation_metrics = {
        **quick_validation,
        "tolerances_declared_before_results": TOLERANCES,
        "analytic_day0": analytic_day0_metrics,
        "scalar_roundtrip": scalar_roundtrip_metrics,
        "velocity_reconstruction": velocity_reconstruction_metrics,
    }
    require_finite_tree(validation_metrics, "validation_metrics")
    validation_metrics["all_reference_preparation_checks_passed"] = True
    checkpoint = {
        "schema_version": 1,
        "run_signature": run_signature,
        "created_at_utc": utc_now(),
        "all_reference_preparation_checks_passed": True,
        "validation_metrics": validation_metrics,
    }
    atomic_write_json(VALIDATION_CHECKPOINT_PATH, checkpoint)
    print(f"Saved validation checkpoint: {VALIDATION_CHECKPOINT_PATH}")
else:
    validation_metrics = checkpoint["validation_metrics"]
    require_finite_tree(validation_metrics, "cached_validation_metrics")
    print(f"Reused validated checkpoint: {VALIDATION_CHECKPOINT_PATH}")


## 8. Mechanically truncate and synthesize T42/T63 products

Each required source scalar is analyzed on the complete MRI Gaussian grid.
Packed coefficients are restricted only by selecting degrees
\(n \le 42\) or \(n \le 63\). No coefficient is tuned, filtered, smoothed,
fitted, or rescaled. Target winds are produced only by the external
library's `getuv` method from the correspondingly restricted vorticity and
divergence spectra.


In [ ]:
def verify_npz(path, spec):
    expected_keys = {
        "time_days", "height", "u", "v",
        "latitude", "longitude", "latitude_weights",
    }
    with np.load(path, allow_pickle=False) as archive:
        keys = set(archive.files)
        if keys != expected_keys:
            raise RuntimeError(f"{path.name}: keys {keys} != {expected_keys}")
        expected_field_shape = (4, spec["nlat"], spec["nlon"])
        expected_shapes = {
            "time_days": (4,),
            "height": expected_field_shape,
            "u": expected_field_shape,
            "v": expected_field_shape,
            "latitude": (spec["nlat"],),
            "longitude": (spec["nlon"],),
            "latitude_weights": (spec["nlat"],),
        }
        for name, shape in expected_shapes.items():
            array = np.asarray(archive[name])
            if array.shape != shape:
                raise RuntimeError(f"{path.name}:{name} shape {array.shape} != {shape}")
            if not np.all(np.isfinite(array)):
                raise RuntimeError(f"{path.name}:{name} contains non-finite values")
            if np.ma.isMaskedArray(array) and np.ma.count_masked(array):
                raise RuntimeError(f"{path.name}:{name} retains masked values")
        if not np.array_equal(archive["time_days"], np.asarray(REQUIRED_TIME_DAYS)):
            raise RuntimeError(f"{path.name}: wrong time_days")
        if not np.all(np.diff(archive["latitude"]) < 0.0):
            raise RuntimeError(f"{path.name}: latitude is not north-to-south")
        periodicity = periodic_longitude_metrics(archive["longitude"])
        if (
            periodicity["max_spacing_error_rad"]
            > TOLERANCES["longitude_periodic_spacing_max_abs_rad"]
            or not periodicity["endpoint_excluded"]
        ):
            raise RuntimeError(f"{path.name}: longitude is not periodic endpoint-excluded")
        if abs(float(archive["latitude_weights"].sum()) - 2.0) > TOLERANCES[
            "final_weight_sum_abs"
        ]:
            raise RuntimeError(f"{path.name}: latitude weights do not sum to 2")
        return {
            "keys": sorted(keys),
            "shapes": {name: list(shape) for name, shape in expected_shapes.items()},
            "dtypes": {name: str(archive[name].dtype) for name in archive.files},
            "longitude_periodicity": periodicity,
            "latitude_orientation": "north_to_south",
            "latitude_weight_sum": float(archive["latitude_weights"].sum()),
        }

def reusable_final_package():
    if not MANIFEST_PATH.exists():
        return None
    try:
        with MANIFEST_PATH.open("r", encoding="utf-8") as stream:
            manifest = strict_json_load(stream)
        if manifest.get("run_signature") != run_signature:
            return None
        if manifest.get("validation", {}).get(
            "all_reference_preparation_checks_passed"
        ) is not True:
            return None
        expected_names = set(TARGETS) | {"manifest.json"}
        actual_names = {path.name for path in OUTPUT_DIR.iterdir() if path.is_file()}
        if actual_names != expected_names:
            return None
        for filename, spec in TARGETS.items():
            path = OUTPUT_DIR / filename
            verify_npz(path, spec)
            if sha256_file(path) != manifest["final_artifacts"][filename]["sha256"]:
                return None
        return manifest
    except Exception:
        return None

manifest = reusable_final_package()
if manifest is None:
    projected = {
        filename: {
            "height": np.empty(
                (len(REQUIRED_TIME_INDICES), spec["nlat"], spec["nlon"]),
                dtype=np.float64,
            ),
            "u": np.empty(
                (len(REQUIRED_TIME_INDICES), spec["nlat"], spec["nlon"]),
                dtype=np.float64,
            ),
            "v": np.empty(
                (len(REQUIRED_TIME_INDICES), spec["nlat"], spec["nlon"]),
                dtype=np.float64,
            ),
        }
        for filename, spec in TARGETS.items()
    }

    with netCDF4.Dataset(SOURCE_CACHE_DIR / "data.nc") as dataset:
        for output_time_index, (source_time_index, time_day) in enumerate(
            zip(REQUIRED_TIME_INDICES, REQUIRED_TIME_DAYS)
        ):
            print(f"Projecting day {time_day:g}...")

            height_source = np.asarray(
                dataset.variables["h"][source_time_index, ::-1, :],
                dtype=np.float64,
            )
            height_full = source_sht.grdtospec(
                height_source, ntrunc=SOURCE_ANALYSIS_TRUNCATION
            )
            del height_source
            for filename, spec in TARGETS.items():
                height_truncated = truncate_spectrum(
                    height_full, spec["truncation"]
                )
                projected[filename]["height"][output_time_index] = (
                    target_sht[filename].spectogrd(height_truncated)
                )
                del height_truncated
            del height_full

            vorticity_source = np.asarray(
                dataset.variables["vor"][source_time_index, ::-1, :],
                dtype=np.float64,
            )
            vorticity_full = source_sht.grdtospec(
                vorticity_source, ntrunc=SOURCE_ANALYSIS_TRUNCATION
            )
            del vorticity_source
            vorticity_truncated = {
                filename: truncate_spectrum(vorticity_full, spec["truncation"])
                for filename, spec in TARGETS.items()
            }
            del vorticity_full

            divergence_source = np.asarray(
                dataset.variables["div"][source_time_index, ::-1, :],
                dtype=np.float64,
            )
            divergence_full = source_sht.grdtospec(
                divergence_source, ntrunc=SOURCE_ANALYSIS_TRUNCATION
            )
            del divergence_source
            for filename, spec in TARGETS.items():
                divergence_truncated = truncate_spectrum(
                    divergence_full, spec["truncation"]
                )
                u_target, v_target = target_sht[filename].getuv(
                    vorticity_truncated[filename], divergence_truncated
                )
                projected[filename]["u"][output_time_index] = u_target
                projected[filename]["v"][output_time_index] = v_target
                del divergence_truncated, u_target, v_target
            del divergence_full, vorticity_truncated

    for filename, arrays in projected.items():
        for field_name in ("height", "u", "v"):
            if not np.all(np.isfinite(arrays[field_name])):
                raise RuntimeError(f"{filename}:{field_name} contains non-finite values")
        coordinates = target_coordinates[filename]
        atomic_save_npz(
            OUTPUT_DIR / filename,
            time_days=np.asarray(REQUIRED_TIME_DAYS, dtype=np.float64),
            height=arrays["height"],
            u=arrays["u"],
            v=arrays["v"],
            latitude=coordinates["latitude"],
            longitude=coordinates["longitude"],
            latitude_weights=coordinates["latitude_weights"],
        )
    del projected
    print("Projected NPZ files written atomically.")
else:
    print("Reused existing final package after hash and content verification.")


## 9. Build the audit manifest and read everything back

The final directory contains only the two minimal NPZ products and
`manifest.json`. Full-resolution spectra, transform work arrays, and
projected vorticity/divergence are not retained.


In [ ]:
if manifest is None:
    notebook_source_hash = None
    notebook_hash_warning = None
    if NOTEBOOK_SOURCE_PATH is not None:
        notebook_path = Path(NOTEBOOK_SOURCE_PATH)
        if not notebook_path.is_file():
            raise RuntimeError(f"NOTEBOOK_SOURCE_PATH is not a file: {notebook_path}")
        notebook_source_hash = sha256_file(notebook_path)
    else:
        notebook_hash_warning = (
            "Notebook byte hash unavailable because Colab does not expose the "
            "executing .ipynb path; set NOTEBOOK_SOURCE_PATH to record it."
        )

    final_artifacts = {}
    for filename, spec in TARGETS.items():
        path = OUTPUT_DIR / filename
        readback = verify_npz(path, spec)
        final_artifacts[filename] = {
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "content_verification": readback,
        }

    caveats = [
        "The published physical fields are float32; projection cannot recover "
        "information absent from that encoding.",
        "MRI weight_lat.nc stores half-normalized Gaussian weights with sum "
        "approximately 1. The fixed factor 2 is applied and equivalence to the "
        "library's sum-2 Gaussian weights is required before transforms.",
        "Skyborn/SPHEREPACK constructs Gaussian quadrature internally and has "
        "no public custom-weight argument. The notebook therefore proves that "
        "its internal weights equal the scaled supplied MRI weights before use.",
        "The archive directory and README call this N959, while STDOUT records "
        "NMAX=958 and NNUM=959. Full source analysis therefore uses the exact "
        "published maximum degree 958; requested T42/T63 restrictions are lower.",
        "Analytic day-zero testing resolves MRI h as layer depth (fluid-layer "
        "thickness). Bottom topography is a separate physical field; free-surface "
        "elevation is layer_depth + bottom_topography.",
        "Reference package v1 mislabeled MRI h as free-surface height. The v2 "
        "semantic correction changes no projected array bytes.",
        "The source archive publishes sizes but not checksums. The notebook "
        "establishes SHA-256 on the first exact-URL HTTPS download and requires "
        "that hash for every later cache reuse.",
        "No full-resolution or target vorticity/divergence spectra, Legendre "
        "tables, Fourier intermediates, or library objects are retained.",
    ]
    if notebook_hash_warning:
        caveats.append(notebook_hash_warning)

    manifest = {
        "schema_version": "mri-w5-reference-v2",
        "created_at_utc": utc_now(),
        "run_signature": run_signature,
        "dataset_identity": {
            "publisher": "Meteorological Research Institute, Japan Meteorological Agency",
            "archive": "Yoshimura DFS shallow-water test cases",
            "test_case": "Williamson shallow-water test case 5",
            "source_directory": SOURCE_DIRECTORY_IDENTITY,
            "source_variant": "sh (spherical-harmonic model output)",
            "published_grid": "1920 longitudes x 960 Gaussian latitudes",
            "archive_label": "N959",
            "stdout_nmax": stdout_nmax,
            "stdout_nnum": stdout_nnum,
            "citation": "Yoshimura (2022), doi:10.5194/gmd-15-2561-2022",
        },
        "source_files": {
            name: source_inventory["files"][name] for name in SOURCE_FILES
        },
        "exact_source_files_used": list(SOURCE_FILES),
        "required_times": {
            "time_indices": list(REQUIRED_TIME_INDICES),
            "time_days": list(REQUIRED_TIME_DAYS),
            "source_coordinate_units": source_time_units,
            "source_coordinate_raw_values": source_time_raw.tolist(),
            "source_coordinate_decoded_values": [str(value) for value in decoded_time],
        },
        "source_inspection": source_inspection,
        "source_variable_mapping": {
            **source_variable_mapping,
            "height": {
                **source_variable_mapping["height"],
                "physical_meaning": "layer_depth",
                "topography_is_separate": True,
            },
        },
        "height_interpretation": {
            "resolved_as": resolved_height_interpretation,
            "meaning": "fluid-layer depth in metres; bottom topography is a separate field",
            "preferred_machine_name": "layer_depth",
            "source_variable": "h",
            "bottom_topography_relationship": (
                "free_surface_elevation = layer_depth + bottom_topography"
            ),
            "analytic_candidate_formulas": {
                "layer_depth": (
                    "h0 - ((a*Omega*u0 + 0.5*u0^2)/g)*sin(latitude)^2"
                ),
                "free_surface_elevation": (
                    "layer_depth + bottom_topography"
                ),
            },
            "analytic_candidate_metrics": height_candidates,
        },
        "external_spherical_harmonic_library": {
            "name": "skyborn",
            "version": SKYBORN_VERSION,
            "backend": "NCAR SPHEREPACK",
            "grid_type": "gaussian",
            "legendre_mode": LEGENDRE_MODE,
            "requested_public_precision": PUBLIC_PRECISION,
            "source_analysis_truncation": SOURCE_ANALYSIS_TRUNCATION,
            "documentation": "https://skyborn.readthedocs.io/en/latest/api/spharm.html",
            "operations_used": ["grdtospec", "spectogrd", "getuv",
                                "gaussian_lats_wts", "getspecindx"],
            "weight_use": (
                "MRI wgt multiplied by exactly 2; equality with the library's "
                "internal Gaussian weights required before transform execution"
            ),
        },
        "software_environment": dependency_versions(),
        "transform_conventions": {
            "coefficient_layout": (
                "packed triangular, m-major: (m=0,n=0..N), "
                "(m=1,n=1..N), ...; size (N+1)(N+2)/2"
            ),
            "legendre_normalization": (
                "integral_0^pi P_nm(theta)^2 sin(theta) dtheta = 1"
            ),
            "colatitude": "theta = pi/2 - latitude",
            "longitude_phase": (
                "Skyborn complex c=a+i*b; SPHEREPACK scalar synthesis uses "
                "a*cos(m*lambda)-b*sin(m*lambda), with Greenwich at lambda=0"
            ),
            "triangular_restriction": (
                "mechanical selection of packed coefficients with total degree "
                "n <= requested truncation; no coefficient changes"
            ),
            "vector_reconstruction": (
                "Skyborn Spharmt.getuv applied directly to truncated scalar "
                "relative-vorticity and divergence spectra; radius=6371220 m"
            ),
            "normalization_or_phase_conversion": "none",
        },
        "validation": {
            **validation_metrics,
            "all_reference_preparation_checks_passed": True,
        },
        "analytic_day0_constants": WILLIAMSON,
        "targets": {
            filename: {
                **spec,
                "field_order": "[time, latitude, longitude]",
                "latitude": "Gaussian, radians, north-to-south",
                "longitude": "[0, 2*pi) radians, uniform, endpoint excluded",
                "latitude_weights": "Gauss-Legendre, sum=2",
                "fields": {
                    "height": "metres; MRI h layer depth (fluid-layer thickness)",
                    "u": "m/s eastward",
                    "v": "m/s northward",
                },
            }
            for filename, spec in TARGETS.items()
        },
        "final_artifacts": final_artifacts,
        "migration_from_v1": {
            "v1_manifest_sha256": V1_MANIFEST_SHA256,
            "correction": (
                "The projected arrays were valid, but MRI h was mislabeled as "
                "free-surface height. It is Williamson-5 layer depth."
            ),
            "projected_arrays_changed": False,
            "t42_sha256": EXPECTED_PROJECTED_ARRAY_HASHES["t42_64x128.npz"],
            "t63_sha256": EXPECTED_PROJECTED_ARRAY_HASHES["t63_96x192.npz"],
        },
        "notebook_source": {
            "configured_path": (
                str(NOTEBOOK_SOURCE_PATH) if NOTEBOOK_SOURCE_PATH is not None else None
            ),
            "sha256": notebook_source_hash,
        },
        "warnings_and_scientific_caveats": caveats,
    }
    require_finite_tree(manifest, "manifest")
    atomic_write_json(MANIFEST_PATH, manifest)

# Final readback: JSON, NPZ hashes/content, and exact directory contents.
with MANIFEST_PATH.open("r", encoding="utf-8") as stream:
    manifest_readback = strict_json_load(stream)
if manifest_readback["run_signature"] != run_signature:
    raise RuntimeError("Manifest readback signature mismatch")
expected_final_names = set(TARGETS) | {"manifest.json"}
actual_final_names = {path.name for path in OUTPUT_DIR.iterdir() if path.is_file()}
if actual_final_names != expected_final_names:
    raise RuntimeError(
        f"Final directory must contain only {sorted(expected_final_names)}; "
        f"found {sorted(actual_final_names)}"
    )
for filename, spec in TARGETS.items():
    path = OUTPUT_DIR / filename
    verify_npz(path, spec)
    if sha256_file(path) != EXPECTED_PROJECTED_ARRAY_HASHES[filename]:
        raise RuntimeError(f"{filename}: projected-array bytes changed from v1")
    if sha256_file(path) != manifest_readback["final_artifacts"][filename]["sha256"]:
        raise RuntimeError(f"{filename}: final hash mismatch")
manifest = manifest_readback
print("Final NPZ and JSON readback verification complete.")


## 10. Final handoff

The next cell intentionally prints only source identity/hashes, external
library identity, validation metrics, the resolved height convention,
final artifact paths/hashes, the manifest path, and unresolved caveats.


In [ ]:
scalar_summary = {}
for variable in ("h", "vor", "div"):
    rows = [
        row for row in manifest["validation"]["scalar_roundtrip"]
        if row["variable"] == variable
    ]
    scalar_summary[variable] = {
        "max_abs_over_required_times": max(row["max_abs"] for row in rows),
        "max_normalized_wrms_over_required_times": max(
            row["normalized_wrms"] for row in rows
        ),
    }

velocity_summary = {}
for component in ("u", "v", "speed"):
    rows = [
        row["components"][component]
        for row in manifest["validation"]["velocity_reconstruction"]
    ]
    velocity_summary[component] = {
        "max_abs_m_s-1_over_required_times": max(row["max_abs"] for row in rows),
        "max_normalized_by_speed_wrms_over_required_times": max(
            row["normalized_by_published_speed_wrms"] for row in rows
        ),
    }

synthetic_max_relative = max(
    mode["coefficient_relative_error"]
    for rows in manifest["validation"]["synthetic_modes"].values()
    for mode in rows
)
constant_max_abs = max(
    row["max_abs"]
    for row in manifest["validation"]["constant_scalar"].values()
)

final_handoff = {
    "source_data": {
        "identity": manifest["dataset_identity"],
        "files": {
            name: {
                "url": entry["url"],
                "size_bytes": entry["size_bytes"],
                "sha256": entry["sha256"],
            }
            for name, entry in manifest["source_files"].items()
        },
    },
    "external_library": manifest["external_spherical_harmonic_library"],
    "validation_metrics": {
        "scalar_roundtrip": scalar_summary,
        "velocity_reconstruction": velocity_summary,
        "analytic_day0": {
            name: manifest["validation"]["analytic_day0"][name]
            for name in ("h", "u", "v", "vor", "div")
        },
        "constant_scalar_max_abs": constant_max_abs,
        "synthetic_mode_max_coefficient_relative_error": synthetic_max_relative,
    },
    "resolved_height_convention": manifest["height_interpretation"],
    "projected_reference_files": {
        filename: {
            "path": entry["path"],
            "sha256": entry["sha256"],
        }
        for filename, entry in manifest["final_artifacts"].items()
    },
    "manifest_path": str(MANIFEST_PATH),
    "unresolved_caveats": manifest["warnings_and_scientific_caveats"],
}
print(json.dumps(final_handoff, indent=2, sort_keys=True))
